# DSC0152 - notebook v68

## Kenapa v68 ada

v66 mengembalikan arsitektur v61 dan hanya menambah training: enam fold seed, dua LightGBM seed, tanpa feature selection. Hasil public terbaiknya, `submission_v66_04_ext0.5_tm30_lgb_lean.csv`, mendapat **0.56424**, di atas 0.56312 milik v61.

v68 melakukan hal yang sama sekali lagi. Arsitekturnya tidak diubah. Yang ditambah hanya partisi: empat fold seed baru, `3, 17, 11, 23`, sehingga totalnya sepuluh. Enam partisi v66 tidak dilatih ulang; skrip mengambil rata-rata tersimpan v66 dan menambahkan empat partisi baru dengan bobot yang sama per seed.

## Mengapa ini bukan stacking

Tidak ada model meta, tidak ada fitur baru, dan tidak ada prediksi model lain yang masuk sebagai fitur. Setiap partisi menghasilkan tujuh kaki yang sama dengan v61/v66, lalu kesepuluh partisi dirata-ratakan. Objek yang berubah hanya jumlah partisi, bukan model atau rumus blend.

In [1]:
import os, sys, json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata, ks_2samp
from sklearn.metrics import roc_auc_score

RUN_TRAINING = False
FOLDS = 10
BASE_PREFIX = 'v66'
BASE_SEED_COUNT = 6
NEW_FOLD_SEEDS = '3,17,11,23'
LGB_SEEDS = '0,1'

HERE = Path.cwd().resolve()
ROOT = HERE
for cand in [HERE, *HERE.parents]:
    if (cand / 'data').is_dir() and (cand / 'agas').is_dir():
        ROOT = cand
        break
AGAS = ROOT / 'agas'
VER = AGAS / 'v68'
OUT = VER / 'output'
TRAINING = VER / 'training'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(AGAS / 'shared'))
os.chdir(ROOT)
print('repo root  ', ROOT)
print('version dir', VER)

repo root   /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026
version dir /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v68


## 1. Verifikasi basis v66

Sebelum training baru, rata-rata enam seed v66 dibaca dan metriknya dicocokkan dengan `summary_v66.json`. Kalau angka ini tidak cocok, penambahan partisi tidak punya basis yang bisa dipercaya.

In [2]:
LEGS = ['fac_full', 'lgb_full', 'lgb_lean', 'lgbx_full', 'cat_full', 'xgb_full', 'linear']
r = lambda v: rankdata(v) / len(v)

z66 = np.load(AGAS / 'v66/output/preds_v66.npz', allow_pickle=True)
summary66 = json.loads((AGAS / 'v66/output/summary_v66.json').read_text())
y66 = pd.read_csv(ROOT / 'data/train_applications.csv',
                  usecols=['application_id', 'target']).set_index(
    'application_id').loc[z66['train_ids']].target.values.astype(int)

print('v66 fold seeds:', summary66['fold_seeds'])
print('v66 lgb seeds :', summary66['lgb_seeds'])
for k in LEGS:
    auc = roc_auc_score(y66, z66[f'oof_{k}'])
    ks = ks_2samp(z66[f'oof_{k}'][y66 == 1], z66[f'oof_{k}'][y66 == 0]).statistic
    print(f'{k:10s} AUC={auc:.6f} KS={ks:.6f} '
          f'summary={summary66[k]["auc"]:.6f}/{summary66[k]["ks"]:.6f}')

v66 fold seeds: [42, 7, 13, 21, 5, 101]
v66 lgb seeds : [0, 1]
fac_full   AUC=0.846662 KS=0.524205 summary=0.846662/0.524205
lgb_full   AUC=0.846682 KS=0.524974 summary=0.846682/0.524974
lgb_lean   AUC=0.846721 KS=0.525311 summary=0.846721/0.525311
lgbx_full  AUC=0.846550 KS=0.524089 summary=0.846550/0.524089
cat_full   AUC=0.846311 KS=0.525234 summary=0.846311/0.525234
xgb_full   AUC=0.846756 KS=0.525536 summary=0.846756/0.525536
linear     AUC=0.842771 KS=0.522712 summary=0.842771/0.522712


## 2. Training v68

Empat partisi baru dijalankan dengan tujuh kaki dan parameter yang identik dengan v66. Skrip menyimpan partial per seed, jadi run bisa dilanjutkan tanpa mengulang seed yang sudah selesai.

In [3]:
cmd = (f'python3 {TRAINING}/train_submission_v68.py '
       f'--folds {FOLDS} --fold-seeds {NEW_FOLD_SEEDS} --lgb-seeds {LGB_SEEDS} '
       f'--base-prefix {BASE_PREFIX} --base-seed-count {BASE_SEED_COUNT} --out-prefix v68')
print(cmd)
if RUN_TRAINING:
    subprocess.run(cmd, shell=True, check=True, cwd=ROOT)

python3 /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v68/training/train_submission_v68.py --folds 10 --fold-seeds 3,17,11,23 --lgb-seeds 0,1 --base-prefix v66 --base-seed-count 6 --out-prefix v68


## 3. Sepuluh seed lawan enam seed

Tabel ini membandingkan kaki OOF yang sama setelah empat partisi ditambahkan. Tidak ada parameter atau himpunan kolom yang berubah, jadi selisihnya adalah efek partisi.

In [4]:
z68 = np.load(OUT / 'preds_v68.npz', allow_pickle=True)
summary68 = json.loads((OUT / 'summary_v68.json').read_text())
assert list(z68['train_ids']) == list(z66['train_ids'])
assert list(z68['test_ids']) == list(z66['test_ids'])
y = pd.read_csv(ROOT / 'data/train_applications.csv',
                usecols=['application_id', 'target']).set_index(
    'application_id').loc[z68['train_ids']].target.values.astype(int)
assert (y == y66).all()

rows = []
for k in LEGS:
    a68 = roc_auc_score(y, z68[f'oof_{k}'])
    k68 = ks_2samp(z68[f'oof_{k}'][y == 1], z68[f'oof_{k}'][y == 0]).statistic
    a66 = roc_auc_score(y, z66[f'oof_{k}'])
    k66 = ks_2samp(z66[f'oof_{k}'][y == 1], z66[f'oof_{k}'][y == 0]).statistic
    rows.append({'leg': k, 'auc_v68': a68, 'auc_v66': a66, 'delta_auc': a68 - a66,
                 'ks_v68': k68, 'ks_v66': k66, 'delta_ks': k68 - k66})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:.6f}'))

      leg  auc_v68  auc_v66  delta_auc   ks_v68   ks_v66  delta_ks
 fac_full 0.846760 0.846662   0.000098 0.524287 0.524205  0.000081
 lgb_full 0.846834 0.846682   0.000153 0.525618 0.524974  0.000644
 lgb_lean 0.846857 0.846721   0.000136 0.526881 0.525311  0.001570
lgbx_full 0.846645 0.846550   0.000094 0.524779 0.524089  0.000690
 cat_full 0.846523 0.846311   0.000212 0.525501 0.525234  0.000267
 xgb_full 0.846891 0.846756   0.000135 0.525990 0.525536  0.000454
   linear 0.842910 0.842771   0.000139 0.524114 0.522712  0.001401


## 4. Rantai blend dan resep winner

Rantai `trees4 -> core -> tm20/tm30 -> ekstrapolasi` dibangun dari sepuluh seed dan dibandingkan dengan rantai yang sama dari enam seed.

In [5]:
def chain(z):
    ro = {k[4:]: r(z[k]) for k in z.files if k.startswith('oof_')}
    trees = (ro['lgb_full'] + ro['lgb_lean'] + ro['cat_full'] + ro['xgb_full']) / 4
    core = (trees * 2 + ro['lgbx_full']) / 3
    tm20 = 0.8 * core + 0.2 * ro['linear']
    tm30 = 0.7 * core + 0.3 * ro['linear']
    return ro, {'trees4': trees, 'core': core, 'tm20': tm20, 'tm30': tm30}

ro68, ch68 = chain(z68)
ro66, ch66 = chain(z66)
rows = []
for name in ['trees4', 'core', 'tm20', 'tm30']:
    rows.append({'chain': name,
                 'ks_v68': ks_2samp(ch68[name][y == 1], ch68[name][y == 0]).statistic,
                 'ks_v66': ks_2samp(ch66[name][y == 1], ch66[name][y == 0]).statistic,
                 'auc_v68': roc_auc_score(y, ch68[name]),
                 'auc_v66': roc_auc_score(y, ch66[name])})
for base in ['tm20', 'tm30']:
    for leg in ['lgb_lean', 'fac_full']:
        for ratio in [0.5, 0.75]:
            o = r(ch68[base] + ratio * (ch68[base] - ro68[leg]))
            o66 = r(ch66[base] + ratio * (ch66[base] - ro66[leg]))
            rows.append({'chain': f'ext{ratio}|{base}|{leg}',
                         'ks_v68': ks_2samp(o[y == 1], o[y == 0]).statistic,
                         'ks_v66': ks_2samp(o66[y == 1], o66[y == 0]).statistic,
                         'auc_v68': roc_auc_score(y, o),
                         'auc_v66': roc_auc_score(y, o66)})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:.6f}'))

                chain   ks_v68   ks_v66  auc_v68  auc_v66
               trees4 0.526196 0.525693 0.847283 0.847142
                 core 0.525385 0.525719 0.847189 0.847072
                 tm20 0.527124 0.526970 0.847427 0.847312
                 tm30 0.527661 0.528476 0.847334 0.847223
 ext0.5|tm20|lgb_lean 0.528299 0.528390 0.847316 0.847206
ext0.75|tm20|lgb_lean 0.528832 0.528286 0.847164 0.847053
 ext0.5|tm20|fac_full 0.528483 0.528496 0.847327 0.847199
ext0.75|tm20|fac_full 0.528882 0.529101 0.847174 0.847031
 ext0.5|tm30|lgb_lean 0.528390 0.528713 0.846956 0.846840
ext0.75|tm30|lgb_lean 0.528243 0.527792 0.846606 0.846490
 ext0.5|tm30|fac_full 0.529355 0.528889 0.846928 0.846797
ext0.75|tm30|fac_full 0.528561 0.529078 0.846552 0.846403


## 5. Kandidat dan validasi

Kandidat ditulis oleh `make_submission_v68.py`. Semua file harus 30000 baris, ID unik dan sama dengan test, serta probabilitas di `[0, 1]`.

In [6]:
cand = json.loads((OUT / 'summary_v68_candidates.json').read_text())
df = pd.DataFrame(cand['candidates'])
print(df[['slot', 'name', 'auc', 'ks', 'ks_minus_sigma', 'spearman_v66_anchor']].to_string(index=False))
test_ids = pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id']).application_id.values
for row in cand['candidates']:
    d = pd.read_csv(OUT / row['file'])
    ok = (len(d) == 30000 and d.application_id.is_unique
          and list(d.application_id) == list(test_ids)
          and d.default_probability.between(0, 1).all()
          and not d.default_probability.isna().any())
    assert ok, row['file']
print(f"validated {len(cand['candidates'])} files against data/test_applications.csv")

 slot                  name      auc       ks  ks_minus_sigma  spearman_v66_anchor
    1                  tm20 0.847427 0.527124        0.523524             0.998256
    2 plateau|tm20|fac_full 0.847328 0.528539        0.524939             0.999154
    3                trees4 0.847283 0.526196        0.522596             0.995121
    4  ext0.5|tm30|lgb_lean 0.846956 0.528390        0.524790             0.999997
    5    ext1|tm20|fac_full 0.846950 0.529867        0.526267             0.998987
    6          leg|xgb_full 0.846891 0.525990        0.522390             0.994739
    7          leg|lgb_lean 0.846857 0.526881        0.523281             0.993050
    8          leg|lgb_full 0.846834 0.525618        0.522018             0.993801
    9         leg|lgbx_full 0.846645 0.524779        0.521179             0.993123
   10          leg|cat_full 0.846523 0.525501        0.521901             0.990684
   11  ext1.5|tm20|lgb_lean 0.846304 0.528175        0.524575             0.998811
   1

validated 14 files against data/test_applications.csv


## 6. Kesimpulan

v68 bukan arsitektur baru dan bukan stacking. Ini resep v61/v66 yang sama dengan sepuluh partisi, bukan enam. Kandidat yang dipertahankan secara paksa tetap `ext0.5|tm30|lgb_lean`, arah yang menghasilkan public 0.56424 di v66. Kandidat lain hanya boleh masuk kalau AUC-nya dekat dan korelasi rank-nya di bawah 0.9995, supaya yang dikirim bukan salinan hampir identik.

Belum ada submission Kaggle dan belum ada push GitHub.

### Hasil aktual sepuluh seed

- Ketujuh kaki naik AUC dibanding v66; `lgb_lean` naik paling besar di KS, +0.001570.
- `ext0.5|tm30|lgb_lean` (slot 04) memberi AUC 0.846956 (+0.000116) dan KS 0.528390 (-0.000323). Korelasi rank terhadap v66_04 adalah 0.999997, jadi ini versi sepuluh seed dari vektor yang sudah terbukti, bukan arah baru.
- Kandidat dengan KS OOF tertinggi adalah `ext1|tm20|fac_full` (slot 05), KS 0.529867 dan KS dikurangi satu sd 0.526267.
- Empat belas file lolos validasi 30000 baris dan cocok dengan `data/test_applications.csv`.
